In [ ]:
import jax
import jax.numpy as jnp
from transformers import AutoTokenizer
from tunix.generate import sampler
from tunix.models.gemma4 import model
from tunix.models.gemma4 import params_safetensors


MESH = [(1, 4), ("fsdp", "tp")]
mesh = jax.make_mesh(
    *MESH, axis_types=(jax.sharding.AxisType.Auto,) * len(MESH[0])
)

# Download HF safetensors to your local dir.
# e.g. hf download google/gemma-4-31B-it --local-dir <your local path>
E2B_PATH = "<your local path>"
E4B_PATH = "<your local path>"
MOE_26B_PATH = "<your local path>"
DENSE_31B_PATH = "<your local path>"

version = "31b"
if version == "e2b":
  tokenizer = AutoTokenizer.from_pretrained(E2B_PATH)
  config = model.ModelConfig.gemma4_e2b()
  model = params_safetensors.create_model_from_safe_tensors(
      E2B_PATH, config, mesh, dtype=jnp.bfloat16
  )
  cache_config = sampler.CacheConfig(
      cache_size=2048, num_layers=35, num_kv_heads=1, head_dim=512
  )
elif version == "e4b":
  tokenizer = AutoTokenizer.from_pretrained(E4B_PATH)
  config = model.ModelConfig.gemma4_e4b()
  model = params_safetensors.create_model_from_safe_tensors(
      E4B_PATH, config, mesh, dtype=jnp.bfloat16
  )
  cache_config = sampler.CacheConfig(
      cache_size=2048, num_layers=42, num_kv_heads=2, head_dim=512
  )
elif version == "moe_26b":
  tokenizer = AutoTokenizer.from_pretrained(MOE_26B_PATH)
  config = model.ModelConfig.gemma4_26b_a4b()
  model = params_safetensors.create_model_from_safe_tensors(MOE_26B_PATH, config, mesh, dtype=jnp.bfloat16)
  cache_config = sampler.CacheConfig(
      cache_size=2048, num_layers=30, num_kv_heads=8, head_dim=512
  )
elif version == "31b":
  tokenizer = AutoTokenizer.from_pretrained(DENSE_31B_PATH)
  config = model.ModelConfig.gemma4_31b()
  model = params_safetensors.create_model_from_safe_tensors(
      DENSE_31B_PATH, config, mesh, dtype=jnp.bfloat16
  )
  cache_config = sampler.CacheConfig(
      cache_size=2048, num_layers=60, num_kv_heads=16, head_dim=512
  )
else:
  raise ValueError(f"Unknown version: {version}")

In [ ]:
def templatize(prompts):
  out = []
  for p in prompts:
    out.append(
        tokenizer.apply_chat_template(
            [
                {"role": "user", "content": p},
            ],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    )
  return out


inputs = templatize([
    "which is larger 9.9 or 9.11?",
    "why sky is blue?",
    "write code to solve fibonacci in python",
    "Write a short joke about saving RAM.",
])

sampler = sampler.Sampler(
    model,
    tokenizer,
    cache_config=cache_config,
)
with mesh:
  out = sampler(
      inputs,
      max_generation_steps=1024,
      echo=True,
      temperature=0.6,
      eos_tokens=[1, 106, 50],
  )

  for t in out.text:
    print(t)
    print("*" * 30)